# STAC to Filesystem Indexing

**Nostradamus - Cube in a Box Tools Series**

---

*   **Objective:** Convert STAC metadata to a local filesystem structure for datacube indexing.
*   **Products used:** Created by user
*   **Source:** [Planetary Computer Catalog](https://planetarycomputer.microsoft.com/catalog)

---

## Background

Indexing data from STAC APIs often requires local metadata organization. This workflow download data from a defined STAC product, and prepares files for efficient datacube ingestion.

## Description

This notebook walks through identifying STAC items, downloading data, preparing metadata by configuring local product definitions, and index the product

***

In [1]:
import sys
sys.path.insert(1, '../utils/')

In [2]:
# reload module before executing code
%load_ext autoreload
%autoreload 2
%matplotlib inline

# # Standard library imports
import os
import time
    
from IPython.display import JSON
from pathlib import Path

from utils.nostra_add_product import (
    select_and_save_product,
    add_product_via_cli,
    parse_product_yaml
)
from utils.nostra_mapping import display_crosshair, MapHandler
from utils.nostra_stac_to_filesystem import (
    stac_to_filesystem,
    list_filesystem_tree,
    find_last_level_folders,
    prepare_filesystem_folders,
    dc_add_dataset,
)
from utils.nostra_tools import extract_path_from_string, style_output_cells

## 1. Configure and add a new product

In [3]:
# Configuration

yamls_dict = {
        "local": "/conf",
        "dc_tools": "https://raw.githubusercontent.com/opendatacube/odc-tools/refs/heads/develop/apps/dc_tools/tests/data/example_product_list.csv"
    }

Before indexing, a product description in YAML format needs to be added to the datacube.

The following cell generates a YAML template based on a selection of existing products. You will need to adapt this template to the product you want to add before proceeding with the subsequent cells.

In [4]:
select_and_save_product(yamls_dict, "example.yaml")

In [5]:
# adapted yaml

yaml_file = 'ls89_c2l2_sp_local-product.yaml'
parse_product_yaml(yaml_file)

YAML file is valid!
Product name: ls89_c2l2_sp_local
Description: USGS Landsat 8 and 9 Collection 2 Level-2 Science Products, consisting of atmospherically corrected surface reflectance and surface temperature image data.
Metadata type: eo3
Number of measurements: 3
Measurements:
  1. blue
  2. green
  3. red


In [6]:
# TbD: Print variable and values to be found on yaml

r = add_product_via_cli(yaml_file, update=True)

Success: Adding "ls89_c2l2_sp_local" (this might take a while) DONE



## 2. Identify scenes and  download

In [7]:
# Create an instance of MapHandler
map_handler = MapHandler()
m, dc = map_handler.create_map()
display(m)

# append crosshair
time.sleep(2)  # make sure m is fully displayed
display_crosshair()

Map(center=[0, 0], controls=(AttributionControl(options=['position', 'prefix'], position='bottomright'), Searc…

<IPython.core.display.Javascript object>

In [8]:
# Warn in case of full AoI
aoi_poly = map_handler.aoi_tupple
if aoi_poly is None:
    style_output_cells('salmon', border_color='red', border_width='2px')
    print('The area of interest polygon (aoi_poly) has not been created. Please draw it in the previous cell.')
else:
    style_output_cells()

In [9]:
# Configuration

stac_endpoint = "https://planetarycomputer.microsoft.com/api/stac/v1"
stac_collection = "landsat-c2-l2"
stac_platforms=["landsat-8", "landsat-9"]
stac_time = ('2024-07-15', '2024-08-01')  # end date is not inclusive !
stac_layers = ['red', 'green', 'blue'] # all if empty ['atran', 'blue', 'cdist', 'coastal', 'drad', 'emis', 'emsd', 'green', 'lwir11', 'nir08', 'qa', 'qa_aerosol', 'qa_pixel', 'qa_radsat', 'red', 'swir16', 'swir22', 'trad', 'urad']

# product_type = 'Landsat'
product_name = 'ls89_c2l2_sp_local'  # 'landsat_ot_c2_l2'
yaml_conf = 'landsat89_mtl_config.yaml'

fs_path = '/local_data'

In [10]:
%%time

# Download from STAC to filesystem

assets_dict, download_results = stac_to_filesystem(
    base_path = fs_path,
    stac_endpoint = stac_endpoint, stac_collection = stac_collection,
    platforms=stac_platforms,
    stac_time = stac_time, stac_layers = stac_layers, t1_only = True,
    aoi_poly = aoi_poly,
    overwrite=True, complete=True, dry_run=False, verbose=False
)
JSON(download_results) # display summary

- Fetching STAC assets...

Download all_assets


- Done
CPU times: user 7.22 s, sys: 3.52 s, total: 10.7 s
Wall time: 16.1 s


<IPython.core.display.JSON object>

In [11]:
# Optional

list_filesystem_tree(fs_path, 'landsat-c2/level-2/standard/oli-tirs',
                 max_recursion_level=4, show_sizes=True)

└── 2024 (2.98 GB)
    ├── 195 (2.06 GB)
    │   ├── 028 (535.14 MB)
    │   │   ├── LC09_L2SP_195028_20240729_20240730_02_T1 (263.20 MB)
    │   │   │   ├── LC09_L2SP_195028_20240729_20240730_02_T1_ANG.txt (114.52 KB)
    │   │   │   ├── LC09_L2SP_195028_20240729_20240730_02_T1_SR_B2.TIF (85.82 MB)
    │   │   │   ├── LC09_L2SP_195028_20240729_20240730_02_T1_SR_B4.TIF (88.85 MB)
    │   │   │   ├── LC09_L2SP_195028_20240729_20240730_02_T1_MTL.txt (14.89 KB)
    │   │   │   ├── LC09_L2SP_195028_20240729_20240730_02_T1_SR_B3.TIF (88.37 MB)
    │   │   │   ├── LC09_L2SP_195028_20240729_20240730_02_T1_MTL.xml (21.93 KB)
    │   │   │   ├── LC09_L2SP_195028_20240729_20240730_02_T1_MTL.json (14.04 KB)
    │   │   │   └── ls89_c2l2_sp_local-metadata.yaml (2.50 KB)
    │   │   └── LC08_L2SP_195028_20240721_20240723_02_T1 (271.94 MB)
    │   │       ├── LC08_L2SP_195028_20240721_20240723_02_T1_ANG.txt (114.52 KB)
    │   │       ├── LC08_L2SP_195028_20240721_20240723_02_T1_SR_B4.TIF (90.65 MB)

## 3. Prepare metadata (.yamls)

In [12]:
# List scenes to process

# OPTION 1: from stac_to_filesystem output
folders = list(assets_dict.keys())

# # OPTION 2: all folders on filesystem
# folders = find_last_level_folders(fs_path)  # "*/oli-tirs/**/*2023072*"

# recreate absolute path
folders = [os.path.join(fs_path, folder) for folder in folders]
folders

['/local_data/landsat-c2/level-2/standard/oli-tirs/2024/194/030/LC08_L2SP_194030_20240730_20240801_02_T1',
 '/local_data/landsat-c2/level-2/standard/oli-tirs/2024/195/030/LC09_L2SP_195030_20240729_20240730_02_T1',
 '/local_data/landsat-c2/level-2/standard/oli-tirs/2024/195/029/LC09_L2SP_195029_20240729_20240730_02_T1',
 '/local_data/landsat-c2/level-2/standard/oli-tirs/2024/194/030/LC09_L2SP_194030_20240722_20240723_02_T1',
 '/local_data/landsat-c2/level-2/standard/oli-tirs/2024/195/030/LC08_L2SP_195030_20240721_20240723_02_T1',
 '/local_data/landsat-c2/level-2/standard/oli-tirs/2024/195/029/LC08_L2SP_195029_20240721_20240723_02_T1']

In [13]:
# optionally remove existing yamls

for folder_path in folders:
    folder = Path(folder_path)
    if folder.is_dir():
        for yaml_file in folder.glob("*.yaml"):
            yaml_file.unlink()  # This deletes the file
            print(f"Deleted: {yaml_file}")
folders

['/local_data/landsat-c2/level-2/standard/oli-tirs/2024/194/030/LC08_L2SP_194030_20240730_20240801_02_T1',
 '/local_data/landsat-c2/level-2/standard/oli-tirs/2024/195/030/LC09_L2SP_195030_20240729_20240730_02_T1',
 '/local_data/landsat-c2/level-2/standard/oli-tirs/2024/195/029/LC09_L2SP_195029_20240729_20240730_02_T1',
 '/local_data/landsat-c2/level-2/standard/oli-tirs/2024/194/030/LC09_L2SP_194030_20240722_20240723_02_T1',
 '/local_data/landsat-c2/level-2/standard/oli-tirs/2024/195/030/LC08_L2SP_195030_20240721_20240723_02_T1',
 '/local_data/landsat-c2/level-2/standard/oli-tirs/2024/195/029/LC08_L2SP_195029_20240721_20240723_02_T1']

In [14]:
results = prepare_filesystem_folders(folders, product_name, yaml_conf)
results

Processing 6 folders with 8 threads...


Processing:   0%|          | 0/6 [00:00<?, ?it/s]


✓ Successful: 6, ✗ Failed: 0


[{'folder': '/local_data/landsat-c2/level-2/standard/oli-tirs/2024/194/030/LC08_L2SP_194030_20240730_20240801_02_T1',
  'result': '✅ YAML document saved to /local_data/landsat-c2/level-2/standard/oli-tirs/2024/194/030/LC08_L2SP_194030_20240730_20240801_02_T1/ls89_c2l2_sp_local-metadata.yaml',
  'status': 'success'},
 {'folder': '/local_data/landsat-c2/level-2/standard/oli-tirs/2024/194/030/LC09_L2SP_194030_20240722_20240723_02_T1',
  'result': '✅ YAML document saved to /local_data/landsat-c2/level-2/standard/oli-tirs/2024/194/030/LC09_L2SP_194030_20240722_20240723_02_T1/ls89_c2l2_sp_local-metadata.yaml',
  'status': 'success'},
 {'folder': '/local_data/landsat-c2/level-2/standard/oli-tirs/2024/195/029/LC09_L2SP_195029_20240729_20240730_02_T1',
  'result': '✅ YAML document saved to /local_data/landsat-c2/level-2/standard/oli-tirs/2024/195/029/LC09_L2SP_195029_20240729_20240730_02_T1/ls89_c2l2_sp_local-metadata.yaml',
  'status': 'success'},
 {'folder': '/local_data/landsat-c2/level-2/st

## 4. Index scenes

In [15]:
yamls =[extract_path_from_string(r['result']) for r in results]
yamls

['/local_data/landsat-c2/level-2/standard/oli-tirs/2024/194/030/LC08_L2SP_194030_20240730_20240801_02_T1/ls89_c2l2_sp_local-metadata.yaml',
 '/local_data/landsat-c2/level-2/standard/oli-tirs/2024/194/030/LC09_L2SP_194030_20240722_20240723_02_T1/ls89_c2l2_sp_local-metadata.yaml',
 '/local_data/landsat-c2/level-2/standard/oli-tirs/2024/195/029/LC09_L2SP_195029_20240729_20240730_02_T1/ls89_c2l2_sp_local-metadata.yaml',
 '/local_data/landsat-c2/level-2/standard/oli-tirs/2024/195/030/LC09_L2SP_195030_20240729_20240730_02_T1/ls89_c2l2_sp_local-metadata.yaml',
 '/local_data/landsat-c2/level-2/standard/oli-tirs/2024/195/030/LC08_L2SP_195030_20240721_20240723_02_T1/ls89_c2l2_sp_local-metadata.yaml',
 '/local_data/landsat-c2/level-2/standard/oli-tirs/2024/195/029/LC08_L2SP_195029_20240721_20240723_02_T1/ls89_c2l2_sp_local-metadata.yaml']

In [16]:
newly_added, already_indexed, failed = dc_add_dataset(yamls, max_workers=4)
    
print(f"\n{'='*60}")
print(f"Summary:")
print(f"  Newly added: {len(newly_added)}")
print(f"  Already indexed: {len(already_indexed)}")
print(f"  Failed: {len(failed)}")
print(f"  Total: {len(yamls)}")
print(f"{'='*60}")

Starting to add 6 datasets with 4 workers...
✓ Successfully added: /local_data/landsat-c2/level-2/standard/oli-tirs/2024/195/029/LC09_L2SP_195029_20240729_20240730_02_T1/ls89_c2l2_sp_local-metadata.yaml
✓ Successfully added: /local_data/landsat-c2/level-2/standard/oli-tirs/2024/195/030/LC09_L2SP_195030_20240729_20240730_02_T1/ls89_c2l2_sp_local-metadata.yaml
✓ Successfully added: /local_data/landsat-c2/level-2/standard/oli-tirs/2024/194/030/LC09_L2SP_194030_20240722_20240723_02_T1/ls89_c2l2_sp_local-metadata.yaml
✓ Successfully added: /local_data/landsat-c2/level-2/standard/oli-tirs/2024/194/030/LC08_L2SP_194030_20240730_20240801_02_T1/ls89_c2l2_sp_local-metadata.yaml
✓ Successfully added: /local_data/landsat-c2/level-2/standard/oli-tirs/2024/195/029/LC08_L2SP_195029_20240721_20240723_02_T1/ls89_c2l2_sp_local-metadata.yaml
✓ Successfully added: /local_data/landsat-c2/level-2/standard/oli-tirs/2024/195/030/LC08_L2SP_195030_20240721_20240723_02_T1/ls89_c2l2_sp_local-metadata.yaml

Comple

You can test data were properly indexed by using [Test_fs_indexation.ipynb](Test_fs_indexation.ipynb).

In [20]:
style_output_cells('salmon', border_color='red', border_width='2px')
print('Even if the dataset is now available from Jupyter Notebooks.\nYou need to ask the CiaB admin to update the Explorer !')

Even if the dataset is now available from Jupyter Notebooks.
You need to ask the CiaB admin to update the Explorer !


***

## Additional information

**License:** The code in this notebook is licensed under the [Apache License, Version 2.0](https://www.apache.org/licenses/LICENSE-2.0).

**Last tested:**

In [21]:
from datetime import datetime
datetime.today().strftime('%Y-%m-%d')

'2026-02-24'

In [22]:
!pip freeze

affine==2.4.0
ai-edge-litert==2.1.2
aiobotocore==3.1.3
aiohappyeyeballs==2.6.1
aiohttp==3.13.3
aioitertools==0.13.0
aiosignal==1.4.0
alembic==1.18.4
annotated-types==0.7.0
antimeridian==0.4.4
anyio==4.12.1
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
arrow==1.4.0
asttokens==3.0.1
async-lru==2.2.0
attrs==25.4.0
babel==2.18.0
backports.strenum==1.2.8
beautifulsoup4==4.14.3
bleach==6.3.0
bokeh==3.8.2
boltons==25.0.0
boto3==1.42.49
botocore==1.42.49
branca==0.8.2
cachetools==7.0.1
cattrs==26.1.0
certifi==2026.1.4
certipy==0.2.2
cffi==2.0.0
cftime==1.6.5
charset-normalizer==3.4.4
ciso8601==2.3.3
click==8.3.1
click-plugins==1.1.1.2
cligj==0.7.2
cloudpickle==3.1.2
comm==0.2.3
contourpy==1.3.3
cryptography==46.0.5
cycler==0.12.1
dask==2025.7.0
dask-image==2025.11.0
dask_labextension==7.0.0
datacube==1.9.14
datadog==0.52.1
debugpy==1.8.20
decorator==5.2.1
defusedxml==0.7.1
deprecat==2.1.3
distributed==2025.7.0
eodatasets3==1.9.3
executing==2.2.1
fastjsonschema==2.21.2
fiona==1.10.1
flatbuff